In [1]:
# Cell 1: Environment setup (install + imports)

# If parquet engine is missing, uncomment:
# %pip install -q pyarrow

from __future__ import annotations

import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [2]:
# Cell 2: Config

@dataclass(frozen=True)
class DataPaths:
    """Container for dataset file paths."""
    features_path: Path
    labels_path: Path
    ohlc_path: Optional[Path] = None


@dataclass(frozen=True)
class FractalConfig:
    """Fractal labeling configuration."""
    left: int
    right: int


@dataclass(frozen=True)
class SplitConfig:
    """Time split configuration."""
    train_ratio: float = 0.70
    valid_ratio: float = 0.15
    test_ratio: float = 0.15


RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DATA_PATHS = DataPaths(
    features_path=Path("/mnt/data/ETHUSDT_15m_features_reduced.parquet"),
    labels_path=Path("/mnt/data/ETHUSDT_15m_fractal_labels_L10_R10.parquet"),
    ohlc_path=Path("/mnt/data/ETHUSDT_15m_ohlc_clean.parquet"),
)

FRACTAL_CFG = FractalConfig(left=10, right=10)
SPLIT_CFG = SplitConfig()

ARTIFACT_DIR = Path("artifacts/ml_fractal_v1_5")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
# Cell 3: IO helpers

def _infer_time_column(dataframe: pd.DataFrame) -> Optional[str]:
    """
    Infer a time column name from common candidates.

    Args:
        dataframe: input dataframe.

    Returns:
        Column name if found, otherwise None.
    """
    candidates = ["timestamp", "open_time", "time", "datetime", "date"]
    for name in candidates:
        if name in dataframe.columns:
            return name
    return None


def load_parquet(path: Path) -> pd.DataFrame:
    """
    Load a parquet file into a DataFrame.

    Args:
        path: parquet file path.

    Returns:
        Loaded dataframe.

    Raises:
        FileNotFoundError: if path does not exist.
    """
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    return pd.read_parquet(path)


def set_time_index(dataframe: pd.DataFrame) -> pd.DataFrame:
    """
    Set a datetime index using an inferred time column if present.

    Args:
        dataframe: input dataframe.

    Returns:
        DataFrame with a datetime index (if possible).
    """
    time_col = _infer_time_column(dataframe)
    if time_col is None:
        return dataframe

    df = dataframe.copy()
    df[time_col] = pd.to_datetime(df[time_col], utc=True, errors="coerce")
    df = df.dropna(subset=[time_col]).sort_values(time_col)
    df = df.set_index(time_col)
    return df


def align_features_and_labels(
    features: pd.DataFrame,
    labels: pd.DataFrame,
) -> pd.DataFrame:
    """
    Align features and labels by time index.

    Args:
        features: feature dataframe indexed by time.
        labels: labels dataframe indexed by time.

    Returns:
        Joined dataframe with features + label columns.
    """
    features_idx = set_time_index(features)
    labels_idx = set_time_index(labels)
    joined = features_idx.join(labels_idx, how="inner")
    joined = joined.sort_index()
    return joined


In [5]:
from pathlib import Path

DATA_PATHS = DataPaths(
    features_path=Path("ETHUSDT_15m_features_reduced.parquet"),
    labels_path=Path("ETHUSDT_15m_fractal_labels_L10_R10.parquet"),
    ohlc_path=Path("ETHUSDT_15m_ohlc_clean.parquet"),
)

In [6]:
# Cell 4: Load data

features_df = load_parquet(DATA_PATHS.features_path)
labels_df = load_parquet(DATA_PATHS.labels_path)

dataset = align_features_and_labels(features_df, labels_df)

print("Dataset shape:", dataset.shape)
print("Columns (tail):", dataset.columns[-10:].tolist())
print("Index range:", dataset.index.min(), "->", dataset.index.max())


Dataset shape: (139219, 11)
Columns (tail): ['ret_3', 'body', 'upper_wick', 'lower_wick', 'dist_to_roll_max_20', 'dist_to_roll_min_20', 'vol_50', 'segment_id', 'y_high', 'y_low']
Index range: 2021-07-05 12:00:00+00:00 -> 2025-06-28 20:30:00+00:00


In [7]:
# Cell 5: Target/feature selection

def infer_label_columns(dataset: pd.DataFrame) -> Tuple[str, str]:
    """
    Infer label column names for swing high and swing low.

    Args:
        dataset: merged dataset.

    Returns:
        Tuple of (high_label_col, low_label_col).

    Raises:
        ValueError: if columns cannot be inferred.
    """
    candidates = [
        ("y_high", "y_low"),
        ("swing_high", "swing_low"),
        ("fractal_high", "fractal_low"),
        ("label_high", "label_low"),
    ]
    for high_col, low_col in candidates:
        if high_col in dataset.columns and low_col in dataset.columns:
            return high_col, low_col
    raise ValueError("Could not infer label columns (high/low).")


high_label_col, low_label_col = infer_label_columns(dataset)

non_feature_cols = {high_label_col, low_label_col}
feature_cols = [c for c in dataset.columns if c not in non_feature_cols]

X_all = dataset[feature_cols].copy()
y_high_all = dataset[high_label_col].astype(int).copy()
y_low_all = dataset[low_label_col].astype(int).copy()

print("n_features:", len(feature_cols))
print("positive_rate_high:", y_high_all.mean())
print("positive_rate_low:", y_low_all.mean())


n_features: 9
positive_rate_high: 0.03373821102004755
positive_rate_low: 0.03424101595328224


In [8]:
# Cell 6: Time split (no shuffle)

def time_split_indices(
    n_rows: int,
    split_cfg: SplitConfig,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Create chronological train/valid/test indices.

    Args:
        n_rows: number of rows.
        split_cfg: split configuration.

    Returns:
        train_idx, valid_idx, test_idx
    """
    train_end = int(n_rows * split_cfg.train_ratio)
    valid_end = int(n_rows * (split_cfg.train_ratio + split_cfg.valid_ratio))

    idx = np.arange(n_rows)
    train_idx = idx[:train_end]
    valid_idx = idx[train_end:valid_end]
    test_idx = idx[valid_end:]
    return train_idx, valid_idx, test_idx


train_idx, valid_idx, test_idx = time_split_indices(len(dataset), SPLIT_CFG)

X_train, X_valid, X_test = X_all.iloc[train_idx], X_all.iloc[valid_idx], X_all.iloc[test_idx]
y_high_train, y_high_valid, y_high_test = y_high_all.iloc[train_idx], y_high_all.iloc[valid_idx], y_high_all.iloc[test_idx]
y_low_train, y_low_valid, y_low_test = y_low_all.iloc[train_idx], y_low_all.iloc[valid_idx], y_low_all.iloc[test_idx]

print("Splits:", len(X_train), len(X_valid), len(X_test))


Splits: 97453 20883 20883


In [9]:
# Cell 7: Model builders

def build_logreg_model() -> Pipeline:
    """
    Build a logistic regression baseline pipeline.

    Returns:
        Sklearn Pipeline with scaler + logistic regression.
    """
    return Pipeline(
        steps=[
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            (
                "model",
                LogisticRegression(
                    max_iter=500,
                    class_weight="balanced",
                    n_jobs=None,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )


def build_hgb_model() -> HistGradientBoostingClassifier:
    """
    Build a histogram gradient boosting baseline.

    Returns:
        HistGradientBoostingClassifier instance.
    """
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=6,
        max_iter=300,
        random_state=RANDOM_SEED,
    )


In [10]:
# Cell 8: Training + evaluation utilities

def compute_pr_auc(y_true: pd.Series, y_prob: np.ndarray) -> float:
    """
    Compute PR-AUC (Average Precision).

    Args:
        y_true: true labels.
        y_prob: predicted probabilities for positive class.

    Returns:
        PR-AUC score.
    """
    return float(average_precision_score(y_true.values, y_prob))


def find_threshold_max_f1(
    y_true: pd.Series,
    y_prob: np.ndarray,
) -> float:
    """
    Find threshold that maximizes F1-score.

    Args:
        y_true: true labels.
        y_prob: predicted probabilities.

    Returns:
        Best threshold.
    """
    precision, recall, thresholds = precision_recall_curve(y_true.values, y_prob)
    # thresholds has length n-1; align by skipping the first precision/recall
    f1_scores = (2 * precision[1:] * recall[1:]) / (precision[1:] + recall[1:] + 1e-12)
    best_idx = int(np.nanargmax(f1_scores))
    return float(thresholds[best_idx])


def print_basic_report(
    y_true: pd.Series,
    y_prob: np.ndarray,
    threshold: float,
    title: str,
) -> None:
    """
    Print PR-AUC and classification report at a given threshold.

    Args:
        y_true: true labels.
        y_prob: predicted probabilities.
        threshold: decision threshold.
        title: report title.
    """
    y_pred = (y_prob >= threshold).astype(int)
    pr_auc = compute_pr_auc(y_true, y_prob)

    print("=" * 80)
    print(title)
    print("PR-AUC:", pr_auc)
    print("Threshold:", threshold)
    print("Confusion matrix:\n", confusion_matrix(y_true.values, y_pred))
    print(classification_report(y_true.values, y_pred, digits=4))


In [11]:
# Cell 9: Online confirmation simulation (delay R)

def simulate_confirmation(
    y_prob: np.ndarray,
    threshold: float,
    right_delay: int,
) -> np.ndarray:
    """
    Simulate online confirmation: a prediction at time t is confirmed at t+R.

    Args:
        y_prob: predicted probabilities aligned to time t.
        threshold: decision threshold applied at time t.
        right_delay: R (confirmation delay).

    Returns:
        confirmed_pred: array aligned to original index where confirmed_pred[t]
            indicates "a swing confirmed at time t" (i.e., originated at t-R).
    """
    raw_pred = (y_prob >= threshold).astype(int)
    confirmed = np.zeros_like(raw_pred)

    if right_delay <= 0:
        return raw_pred

    confirmed[right_delay:] = raw_pred[:-right_delay]
    return confirmed


In [12]:
# Cell 10: Train + evaluate for HIGH (repeat for LOW)

high_logreg = build_logreg_model()
high_logreg.fit(X_train, y_high_train)

high_valid_prob = high_logreg.predict_proba(X_valid)[:, 1]
high_threshold = find_threshold_max_f1(y_high_valid, high_valid_prob)

print_basic_report(
    y_true=y_high_valid,
    y_prob=high_valid_prob,
    threshold=high_threshold,
    title="HIGH / LogisticRegression / VALID",
)

high_test_prob = high_logreg.predict_proba(X_test)[:, 1]
print_basic_report(
    y_true=y_high_test,
    y_prob=high_test_prob,
    threshold=high_threshold,
    title="HIGH / LogisticRegression / TEST",
)

high_test_confirmed = simulate_confirmation(
    y_prob=high_test_prob,
    threshold=high_threshold,
    right_delay=FRACTAL_CFG.right,
)
print("Confirmed positive rate (test):", high_test_confirmed.mean())


HIGH / LogisticRegression / VALID
PR-AUC: 0.2579895739096428
Threshold: 0.7520163391941913
Confusion matrix:
 [[19050  1123]
 [  334   376]]
              precision    recall  f1-score   support

           0     0.9828    0.9443    0.9632     20173
           1     0.2508    0.5296    0.3404       710

    accuracy                         0.9302     20883
   macro avg     0.6168    0.7370    0.6518     20883
weighted avg     0.9579    0.9302    0.9420     20883

HIGH / LogisticRegression / TEST
PR-AUC: 0.25673895997701657
Threshold: 0.7520163391941913
Confusion matrix:
 [[18748  1459]
 [  281   395]]
              precision    recall  f1-score   support

           0     0.9852    0.9278    0.9557     20207
           1     0.2131    0.5843    0.3123       676

    accuracy                         0.9167     20883
   macro avg     0.5991    0.7561    0.6340     20883
weighted avg     0.9602    0.9167    0.9348     20883

Confirmed positive rate (test): 0.08878034765119953


In [13]:
# Cell 11: Repeat for LOW

low_logreg = build_logreg_model()
low_logreg.fit(X_train, y_low_train)

low_valid_prob = low_logreg.predict_proba(X_valid)[:, 1]
low_threshold = find_threshold_max_f1(y_low_valid, low_valid_prob)

print_basic_report(
    y_true=y_low_valid,
    y_prob=low_valid_prob,
    threshold=low_threshold,
    title="LOW / LogisticRegression / VALID",
)

low_test_prob = low_logreg.predict_proba(X_test)[:, 1]
print_basic_report(
    y_true=y_low_test,
    y_prob=low_test_prob,
    threshold=low_threshold,
    title="LOW / LogisticRegression / TEST",
)

low_test_confirmed = simulate_confirmation(
    y_prob=low_test_prob,
    threshold=low_threshold,
    right_delay=FRACTAL_CFG.right,
)
print("Confirmed positive rate (test):", low_test_confirmed.mean())


LOW / LogisticRegression / VALID
PR-AUC: 0.2920717224545651
Threshold: 0.8168368236826667
Confusion matrix:
 [[19372   814]
 [  362   335]]
              precision    recall  f1-score   support

           0     0.9817    0.9597    0.9705     20186
           1     0.2916    0.4806    0.3629       697

    accuracy                         0.9437     20883
   macro avg     0.6366    0.7202    0.6667     20883
weighted avg     0.9586    0.9437    0.9503     20883

LOW / LogisticRegression / TEST
PR-AUC: 0.2742901127896944
Threshold: 0.8168368236826667
Confusion matrix:
 [[19053  1137]
 [  301   392]]
              precision    recall  f1-score   support

           0     0.9844    0.9437    0.9636     20190
           1     0.2564    0.5657    0.3528       693

    accuracy                         0.9311     20883
   macro avg     0.6204    0.7547    0.6582     20883
weighted avg     0.9603    0.9311    0.9434     20883

Confirmed positive rate (test): 0.07321744960015324


In [14]:
# Cell 12: Optional stronger baseline (HGB) - HIGH/LOW

def compute_sample_weights(y: pd.Series) -> np.ndarray:
    """
    Compute simple inverse-frequency sample weights for binary targets.

    Args:
        y: binary labels.

    Returns:
        Sample weights array.
    """
    pos_rate = float(y.mean())
    pos_rate = max(min(pos_rate, 1.0 - 1e-6), 1e-6)
    w_pos = 0.5 / pos_rate
    w_neg = 0.5 / (1.0 - pos_rate)
    return np.where(y.values == 1, w_pos, w_neg)


hgb_high = build_hgb_model()
hgb_high.fit(X_train, y_high_train, sample_weight=compute_sample_weights(y_high_train))

hgb_high_valid_prob = hgb_high.predict_proba(X_valid)[:, 1]
hgb_high_threshold = find_threshold_max_f1(y_high_valid, hgb_high_valid_prob)

print_basic_report(
    y_true=y_high_valid,
    y_prob=hgb_high_valid_prob,
    threshold=hgb_high_threshold,
    title="HIGH / HistGradientBoosting / VALID",
)


HIGH / HistGradientBoosting / VALID
PR-AUC: 0.3131284908705368
Threshold: 0.8431903444646942
Confusion matrix:
 [[19279   894]
 [  332   378]]
              precision    recall  f1-score   support

           0     0.9831    0.9557    0.9692     20173
           1     0.2972    0.5324    0.3814       710

    accuracy                         0.9413     20883
   macro avg     0.6401    0.7440    0.6753     20883
weighted avg     0.9598    0.9413    0.9492     20883



In [15]:
# Cell 13: Save artifacts

import joblib


def save_json(path: Path, payload: Dict) -> None:
    """
    Save a dictionary as JSON.

    Args:
        path: output path.
        payload: dict to save.
    """
    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, ensure_ascii=False)


# Choose which models you want to ship as baseline (example: logistic regression)
joblib.dump(high_logreg, ARTIFACT_DIR / "model_high.pkl")
joblib.dump(low_logreg, ARTIFACT_DIR / "model_low.pkl")

save_json(
    ARTIFACT_DIR / "feature_config.json",
    {
        "features": feature_cols,
        "n_features": len(feature_cols),
    },
)

save_json(
    ARTIFACT_DIR / "thresholds.json",
    {
        "threshold_high": high_threshold,
        "threshold_low": low_threshold,
        "left": FRACTAL_CFG.left,
        "right": FRACTAL_CFG.right,
        "selection_rule": "max_f1_on_valid",
    },
)

save_json(
    ARTIFACT_DIR / "meta.json",
    {
        "symbol": "ETHUSDT",
        "timeframe": "15m",
        "labels": f"fractal_L{FRACTAL_CFG.left}_R{FRACTAL_CFG.right}",
        "random_seed": RANDOM_SEED,
        "features_source": str(DATA_PATHS.features_path),
        "labels_source": str(DATA_PATHS.labels_path),
    },
)

print("Saved to:", ARTIFACT_DIR.resolve())


Saved to: C:\Users\artkh\Downloads\Ml v1\artifacts\ml_fractal_v1_5
